# Phase C1 — Data Normalization + Data Quality Audit

C1 audits the actual Dataset V1 CSV schema, checks structural/data-quality issues,
and creates conservative normalized copies.

C1 does NOT perform:
- full EDA
- final feature selection
- outlier removal
- imputation
- train/validation/test splitting
- model training

Raw files under `data/generated_csv/` are never modified.

In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

CURRENT = Path.cwd()

if (CURRENT / "data" / "generated_csv").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data" / "generated_csv").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError(
        "Could not locate project root containing data/generated_csv."
    )

DATA_DIR = PROJECT_ROOT / "data" / "generated_csv"
NORM_DIR = PROJECT_ROOT / "data" / "normalized"
REPORT_DIR = PROJECT_ROOT / "reports" / "data_quality"
DOCS_DIR = PROJECT_ROOT / "docs"

NORM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

TABLE_FILES = [
    "customers.csv",
    "subscriptions.csv",
    "payment_attempts.csv",
    "recovery_cases.csv",
    "recovery_events.csv",
    "action_log.csv",
    "case_outcomes.csv",
    "ml_datamart.csv",
]

dfs = {}
for filename in TABLE_FILES:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required table: {path}")
    dfs[path.stem] = pd.read_csv(path, low_memory=False)

print("Loaded:")
for name, df in dfs.items():
    print(f"  {name:20s} {len(df):>10,} rows x {len(df.columns)} cols")


Loaded:
  customers                20,000 rows x 5 cols
  subscriptions            22,078 rows x 12 cols
  payment_attempts        264,936 rows x 11 cols
  recovery_cases          103,026 rows x 19 cols
  recovery_events         986,365 rows x 9 cols
  action_log              157,423 rows x 20 cols
  case_outcomes           103,026 rows x 17 cols
  ml_datamart             157,423 rows x 45 cols


## 1. Table-Level Quality Summary

In [2]:

rows = []

for name, df in dfs.items():
    total_cells = df.shape[0] * df.shape[1]
    missing = int(df.isna().sum().sum())

    rows.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": missing,
        "missing_pct_of_cells": round(
            100 * missing / total_cells, 4
        ) if total_cells else 0.0,
        "duplicate_rows": int(df.duplicated().sum()),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024**2), 2
        ),
    })

table_quality = pd.DataFrame(rows)
display(table_quality)
table_quality.to_csv(
    REPORT_DIR / "table_quality_summary.csv", index=False
)


,table,rows,columns,missing_cells,missing_pct_of_cells,duplicate_rows,memory_mb
0,customers,20000,5,0,0.0000,0,5.10
1,subscriptions,22078,12,0,0.0000,0,10.38
2,payment_attempts,264936,11,238138,8.1714,0,58.88
3,recovery_cases,103026,19,68826,3.5160,0,44.10
4,recovery_events,986365,9,0,0.0000,0,443.03
5,action_log,157423,20,203028,6.4485,0,85.64
6,case_outcomes,103026,17,68826,3.9297,0,31.02
7,ml_datamart,157423,45,203028,2.8660,0,165.09


## 2. Actual Schema / Dtype Audit

In [3]:

schema_rows = []

for name, df in dfs.items():
    for col in df.columns:
        schema_rows.append({
            "table": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "unique_count": int(df[col].nunique(dropna=True)),
        })

schema_audit = pd.DataFrame(schema_rows)
display(schema_audit)
schema_audit.to_csv(
    REPORT_DIR / "schema_audit.csv", index=False
)


,table,column,dtype,missing_count,unique_count
0,customers,customer_id,int64,0,20000
1,customers,age_band,object,0,5
2,customers,state,object,0,10
3,customers,locale_pref,object,0,3
4,customers,signed_up_date,object,0,730
...,...,...,...,...,...
133,ml_datamart,exec_day_of_month,int64,0,30
134,ml_datamart,exec_in_salary_cluster,bool,0,2
135,ml_datamart,y,int64,0,2
136,ml_datamart,response_kind,object,101514,5


## 3. Timestamp / Time-Unit Audit

Dataset V1 contains both ISO/date-like fields and simulation-hour fields.
These are audited separately; `_ts_h` fields are not treated as calendar datetimes.

In [4]:

date_like_cols = [
    "signed_up_date",
    "start_date",
    "ts_iso",
    "ts",
    "opened_ts",
    "closed_ts",
]

timestamp_rows = []
for name, df in dfs.items():
    for col in date_like_cols:
        if col not in df.columns:
            continue

        raw = df[col]
        parsed = pd.to_datetime(raw, errors="coerce")

        timestamp_rows.append({
            "table": name,
            "column": col,
            "null_count": int(raw.isna().sum()),
            "parse_failures": int(
                raw.notna().sum() - parsed.notna().sum()
            ),
            "min": parsed.min(),
            "max": parsed.max(),
        })

timestamp_audit = pd.DataFrame(timestamp_rows)
display(timestamp_audit)

time_hour_rows = []
for name, df in dfs.items():
    for col in df.columns:
        if not col.endswith("_h"):
            continue

        numeric = pd.to_numeric(df[col], errors="coerce")
        time_hour_rows.append({
            "table": name,
            "column": col,
            "null_count": int(numeric.isna().sum()),
            "non_numeric_count": int(
                df[col].notna().sum() - numeric.notna().sum()
            ),
            "min_value": numeric.min(),
            "max_value": numeric.max(),
        })

time_hour_audit = pd.DataFrame(time_hour_rows)
display(time_hour_audit)

timestamp_audit.to_csv(
    REPORT_DIR / "timestamp_audit.csv", index=False
)
time_hour_audit.to_csv(
    REPORT_DIR / "simulation_time_audit.csv", index=False
)


,table,column,null_count,parse_failures,min,max
0,customers,signed_up_date,0,0,2022-01-01 00:00:00,2023-12-31 00:00:00
1,subscriptions,start_date,0,0,2022-01-05 00:00:00,2023-12-31 00:00:00
2,payment_attempts,ts_iso,0,0,2024-01-01 05:00:00,2024-12-23 05:00:00
3,recovery_events,ts_iso,0,0,2024-01-01 06:00:00,2025-01-08 02:00:00


,table,column,null_count,non_numeric_count,min_value,max_value
0,payment_attempts,ts_h,0,0,5.0,8573.0
1,recovery_cases,opened_ts_h,0,0,6.0,8574.0
2,recovery_cases,closed_ts_h,0,0,6.0,8954.0
3,recovery_cases,time_to_recovery_h,68826,0,3.0,385.0
4,recovery_events,ts_h,0,0,6.0,8954.0
5,action_log,decision_ts_h,0,0,6.0,8836.0
6,action_log,exec_ts_h,0,0,8.0,8837.0
7,case_outcomes,opened_ts_h,0,0,6.0,8574.0
8,case_outcomes,closed_ts_h,0,0,6.0,8954.0
9,case_outcomes,time_to_recovery_h,68826,0,3.0,385.0


## 4. Monetary / Unit Audit

In [5]:

money_rows = []

for name, df in dfs.items():
    money_cols = [
        c for c in df.columns
        if c.endswith("_inr")
        or "amount" in c.lower()
        or "cost" in c.lower()
        or "risk" in c.lower()
    ]

    for col in money_cols:
        numeric = pd.to_numeric(df[col], errors="coerce")
        money_rows.append({
            "table": name,
            "column": col,
            "non_numeric_count": int(
                df[col].notna().sum() - numeric.notna().sum()
            ),
            "negative_count": int((numeric < 0).sum()),
            "zero_count": int((numeric == 0).sum()),
            "min_value": numeric.min(),
            "max_value": numeric.max(),
        })

money_audit = pd.DataFrame(money_rows)
display(money_audit)
money_audit.to_csv(
    REPORT_DIR / "monetary_audit.csv", index=False
)


,table,column,non_numeric_count,negative_count,zero_count,min_value,max_value
0,subscriptions,plan_amount_inr,0,0,0,49.0,3999.00
1,subscriptions,mandate_max_amount_inr,0,0,0,80.0,6000.00
2,payment_attempts,amount_inr,0,0,0,49.0,3999.00
3,recovery_cases,amount_at_risk_inr,0,0,0,49.0,3999.00
4,recovery_cases,cost_total_inr,0,0,30181,0.0,51.75
5,recovery_cases,recovered_amount_inr,0,0,68826,0.0,3999.00
6,case_outcomes,recovered_amount_inr,0,0,68826,0.0,3999.00
7,case_outcomes,amount_at_risk_inr,0,0,0,49.0,3999.00
8,case_outcomes,cost_total_inr,0,0,30181,0.0,51.75
9,ml_datamart,amount_inr,0,0,0,49.0,3999.00


## 5. String / Category Consistency Audit

In [6]:

cat_rows = []

for name, df in dfs.items():
    for col in df.select_dtypes(include=["object"]).columns:
        s = df[col].dropna().astype(str)
        stripped = s.str.strip()

        cat_rows.append({
            "table": name,
            "column": col,
            "unique_values": int(s.nunique()),
            "empty_strings": int((s == "").sum()),
            "leading_whitespace": int(s.str.startswith(" ").sum()),
            "trailing_whitespace": int(s.str.endswith(" ").sum()),
            "case_insensitive_unique": (
                int(stripped.str.lower().nunique())
                if len(stripped) else 0
            ),
        })

category_audit = pd.DataFrame(cat_rows)
display(category_audit)
category_audit.to_csv(
    REPORT_DIR / "categorical_consistency.csv", index=False
)


,table,column,unique_values,empty_strings,leading_whitespace,trailing_whitespace,case_insensitive_unique
0,customers,age_band,5,0,0,0,5
1,customers,state,10,0,0,0,10
2,customers,locale_pref,3,0,0,0,3
3,customers,signed_up_date,730,0,0,0,730
4,subscriptions,segment,5,0,0,0,5
5,subscriptions,plan_tier,3,0,0,0,3
6,subscriptions,cadence,1,0,0,0,1
7,subscriptions,start_date,704,0,0,0,704
8,subscriptions,rail,2,0,0,0,2
9,subscriptions,bank_name,8,0,0,0,8


## 6. Primary-Key / Grain Audit

Do not assume `case_id` is globally unique across every table.
For policy-specific case tables, inspect `(policy_id, case_id)` where that
dimension exists.

In [7]:

key_specs = {
    "customers": ["customer_id"],
    "subscriptions": ["subscription_id"],
    "payment_attempts": ["attempt_id"],
    "recovery_events": ["event_id"],
    "action_log": ["action_id"],
}

key_rows = []

for table, keys in key_specs.items():
    df = dfs[table]

    for key in keys:
        if key not in df.columns:
            key_rows.append({
                "table": table,
                "key": key,
                "status": "NOT VALIDATED — COLUMN NOT FOUND"
            })
            continue

        s = df[key]
        key_rows.append({
            "table": table,
            "key": key,
            "rows": len(df),
            "null_ids": int(s.isna().sum()),
            "duplicate_ids": int(s.duplicated().sum()),
            "unique_ids": int(s.nunique()),
            "is_unique": bool(
                s.notna().all() and s.nunique() == len(df)
            ),
        })

for table in ["recovery_cases", "case_outcomes"]:
    df = dfs[table]
    if {"policy_id", "case_id"}.issubset(df.columns):
        subset = ["policy_id", "case_id"]
        key_rows.append({
            "table": table,
            "key": "policy_id + case_id",
            "rows": len(df),
            "null_ids": int(df[subset].isna().any(axis=1).sum()),
            "duplicate_ids": int(df.duplicated(subset=subset).sum()),
            "unique_ids": int(df[subset].drop_duplicates().shape[0]),
            "is_unique": bool(
                df[subset].dropna().duplicated().sum() == 0
            ),
        })

ml = dfs["ml_datamart"]
if "action_id" in ml.columns:
    key_rows.append({
        "table": "ml_datamart",
        "key": "action_id",
        "rows": len(ml),
        "null_ids": int(ml["action_id"].isna().sum()),
        "duplicate_ids": int(ml["action_id"].duplicated().sum()),
        "unique_ids": int(ml["action_id"].nunique()),
        "is_unique": bool(
            ml["action_id"].notna().all()
            and ml["action_id"].nunique() == len(ml)
        ),
    })

key_audit = pd.DataFrame(key_rows)
display(key_audit)
key_audit.to_csv(
    REPORT_DIR / "key_integrity.csv", index=False
)


,table,key,rows,null_ids,duplicate_ids,unique_ids,is_unique
0,customers,customer_id,20000,0,0,20000,True
1,subscriptions,subscription_id,22078,0,0,22078,True
2,payment_attempts,attempt_id,264936,0,0,264936,True
3,recovery_events,event_id,986365,0,605907,380458,False
4,action_log,action_id,157423,0,92497,64926,False
5,recovery_cases,policy_id + case_id,103026,0,0,103026,True
6,case_outcomes,policy_id + case_id,103026,0,0,103026,True


## 7. Relational Integrity

For policy-specific case tables, use the policy dimension when present.
Do not silently join policy-specific cases using `case_id` alone.

In [8]:

def fk_result(parent_name, parent_keys, child_name, child_keys):
    parent = dfs[parent_name]
    child = dfs[child_name]

    if not set(parent_keys).issubset(parent.columns):
        return {
            "parent": parent_name,
            "parent_key": "+".join(parent_keys),
            "child": child_name,
            "child_key": "+".join(child_keys),
            "status": "NOT VALIDATED — PARENT COLUMN NOT FOUND",
        }

    if not set(child_keys).issubset(child.columns):
        return {
            "parent": parent_name,
            "parent_key": "+".join(parent_keys),
            "child": child_name,
            "child_key": "+".join(child_keys),
            "status": "NOT VALIDATED — CHILD COLUMN NOT FOUND",
        }

    parent_key_df = parent[parent_keys].drop_duplicates()
    child_key_df = child[child_keys].copy()

    probe = child_key_df.merge(
        parent_key_df.assign(_parent_exists=1),
        left_on=child_keys,
        right_on=parent_keys,
        how="left",
    )

    orphan_rows = int(
        probe["_parent_exists"].isna().sum()
    )

    return {
        "parent": parent_name,
        "parent_key": "+".join(parent_keys),
        "child": child_name,
        "child_key": "+".join(child_keys),
        "status": "VALIDATED",
        "child_rows": len(child),
        "orphan_rows": orphan_rows,
        "orphan_rate": (
            orphan_rows / len(child) if len(child) else 0.0
        ),
    }

fk_checks = [
    ("customers", ["customer_id"], "subscriptions", ["customer_id"]),
    ("subscriptions", ["subscription_id"], "payment_attempts", ["subscription_id"]),
]

fk_results = [
    fk_result(*check)
    for check in fk_checks
]

if "policy_id" in dfs["recovery_cases"].columns:
    policy_case_key = ["policy_id", "case_id"]

    fk_results.extend([
        fk_result("recovery_cases", policy_case_key,
                  "recovery_events", ["policy_id", "case_id"]),
        fk_result("recovery_cases", policy_case_key,
                  "action_log", ["policy_id", "case_id"]),
        fk_result("recovery_cases", policy_case_key,
                  "case_outcomes", ["policy_id", "case_id"]),
        fk_result("recovery_cases", policy_case_key,
                  "ml_datamart", ["logging_policy", "case_id"]),
    ])
else:
    fk_results.extend([
        fk_result("recovery_cases", ["case_id"],
                  "recovery_events", ["case_id"]),
        fk_result("recovery_cases", ["case_id"],
                  "action_log", ["case_id"]),
        fk_result("recovery_cases", ["case_id"],
                  "case_outcomes", ["case_id"]),
        fk_result("recovery_cases", ["case_id"],
                  "ml_datamart", ["case_id"]),
    ])

fk_audit = pd.DataFrame(fk_results)
display(fk_audit)
fk_audit.to_csv(
    REPORT_DIR / "foreign_key_integrity.csv", index=False
)


,parent,parent_key,child,child_key,status,child_rows,orphan_rows,orphan_rate
0,customers,customer_id,subscriptions,customer_id,VALIDATED,22078,0,0.0
1,subscriptions,subscription_id,payment_attempts,subscription_id,VALIDATED,264936,0,0.0
2,recovery_cases,policy_id+case_id,recovery_events,policy_id+case_id,VALIDATED,986365,0,0.0
3,recovery_cases,policy_id+case_id,action_log,policy_id+case_id,VALIDATED,157423,0,0.0
4,recovery_cases,policy_id+case_id,case_outcomes,policy_id+case_id,VALIDATED,103026,0,0.0
5,recovery_cases,policy_id+case_id,ml_datamart,logging_policy+case_id,VALIDATED,157423,0,0.0


## 8. Dataset V1 Constraint Audit

Only constraints already established for Dataset V1 are checked.

In [9]:

constraint_rows = []

cases = dfs["recovery_cases"]
outcomes = dfs["case_outcomes"]

if (
    {"policy_id", "case_id", "amount_at_risk"}.issubset(cases.columns)
    and {"policy_id", "case_id", "recovered_amount_inr"}.issubset(outcomes.columns)
):
    keys = ["policy_id", "case_id"]

    joined = cases[keys + ["amount_at_risk"]].merge(
        outcomes[keys + ["recovered_amount_inr"]],
        on=keys,
        how="left",
        validate="one_to_one",
    )

    violations = int(
        (
            joined["recovered_amount_inr"]
            > joined["amount_at_risk"]
        ).fillna(False).sum()
    )

    constraint_rows.append({
        "constraint": "recovered_amount_inr <= amount_at_risk",
        "violations": violations,
        "status": "PASS" if violations == 0 else "FAIL",
    })

for table_name in ["recovery_cases", "case_outcomes"]:
    df = dfs[table_name]

    for col, limit in [
        ("retries_used", 3),
        ("attempts_used", 3),
        ("contacts_used", 2),
        ("contacts_sent", 2),
    ]:
        if col in df.columns:
            series = pd.to_numeric(
                df[col], errors="coerce"
            )
            violations = int((series > limit).sum())

            constraint_rows.append({
                "constraint": f"{table_name}.{col} <= {limit}",
                "violations": violations,
                "status": "PASS" if violations == 0 else "FAIL",
            })

ml_target = "y" if "y" in ml.columns else "y_intervention"
target_num = pd.to_numeric(
    ml[ml_target], errors="coerce"
)
invalid_target = int(
    (
        target_num.notna()
        & ~target_num.isin([0, 1])
    ).sum()
)

constraint_rows.append({
    "constraint": f"ml_datamart.{ml_target} is binary 0/1",
    "violations": invalid_target,
    "status": "PASS" if invalid_target == 0 else "FAIL",
})

constraint_audit = pd.DataFrame(constraint_rows)
display(constraint_audit)
constraint_audit.to_csv(
    REPORT_DIR / "constraint_audit.csv", index=False
)


,constraint,violations,status
0,recovery_cases.retries_used <= 3,0,PASS
1,recovery_cases.contacts_sent <= 2,0,PASS
2,case_outcomes.retries_used <= 3,0,PASS
3,case_outcomes.contacts_sent <= 2,0,PASS
4,ml_datamart.y is binary 0/1,0,PASS


## 9. ML-Specific Quality Audit

In [10]:

ml_target = "y" if "y" in ml.columns else "y_intervention"
target_num = pd.to_numeric(ml[ml_target], errors="coerce")

ml_rows = [
    {"check": "target_column", "value": ml_target},
    {"check": "target_missing", "value": int(target_num.isna().sum())},
    {"check": "target_positive_rate", "value": float(target_num.mean())},
]

if "action_type" in ml.columns:
    for action, count in ml["action_type"].value_counts(dropna=False).items():
        ml_rows.append({
            "check": f"action_support::{action}",
            "value": int(count),
        })

if "action_id" in ml.columns:
    ml_rows.append({
        "check": "duplicate_action_ids",
        "value": int(ml["action_id"].duplicated().sum()),
    })

if {"case_id", "decision_seq"}.issubset(ml.columns):
    ml_rows.append({
        "check": "duplicate_case_decision_seq",
        "value": int(
            ml.duplicated(
                subset=["case_id", "decision_seq"]
            ).sum()
        ),
    })

ml_quality = pd.DataFrame(ml_rows)
display(ml_quality)
ml_quality.to_csv(
    REPORT_DIR / "ml_quality_audit.csv", index=False
)


,check,value
0,target_column,y
1,target_missing,0
2,target_positive_rate,0.228131
3,action_support::RETRY,101514
4,action_support::SEND_PAYMENT_LINK,30884
5,action_support::SEND_REMINDER,22649
6,action_support::REQUEST_PM_UPDATE,2376
7,duplicate_case_decision_seq,74061


## 10. Conservative Normalization

Only representation-level transformations are applied:
- IDs → strings
- actual date/ISO fields → standardized datetime strings
- whitespace trimming for string fields

No outlier removal, imputation, or final feature selection is performed.

In [11]:

manifest = []

id_cols = {
    "customer_id", "subscription_id", "attempt_id",
    "case_id", "event_id", "action_id"
}

date_cols = {
    "signed_up_date", "start_date", "ts",
    "ts_iso", "opened_ts", "closed_ts"
}

for name, df in dfs.items():
    norm = df.copy()

    for col in norm.columns:

        if col in id_cols:
            old = str(norm[col].dtype)
            norm[col] = norm[col].astype("string")
            manifest.append({
                "table": name,
                "column": col,
                "change": "cast_to_string",
                "from_dtype": old,
                "to_dtype": str(norm[col].dtype),
            })

        elif col in date_cols:
            parsed = pd.to_datetime(
                norm[col], errors="coerce"
            )
            norm[col] = parsed.dt.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
            manifest.append({
                "table": name,
                "column": col,
                "change": "standardize_datetime_representation",
            })

        elif norm[col].dtype == "object":
            before = norm[col].copy()
            norm[col] = (
                norm[col].astype("string").str.strip()
            )

            changed = int(
                (
                    before.fillna("").astype(str)
                    != norm[col].fillna("").astype(str)
                ).sum()
            )

            if changed:
                manifest.append({
                    "table": name,
                    "column": col,
                    "change": "strip_string_whitespace",
                    "rows_changed": changed,
                })

    norm.to_csv(
        NORM_DIR / f"{name}.csv",
        index=False
    )

with open(
    REPORT_DIR / "normalization_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest, f, indent=2, default=str)

print("Normalized copies written to:", NORM_DIR)


Normalized copies written to: c:\Users\Soham\Documents\revenue-recovery\data\normalized


## 11. C1 Boundary

C1 does not:
- remove outliers
- impute missing values
- select final features
- split data
- train models

Review the generated audits. Then proceed to **C2 — Comprehensive EDA**.